In [1]:
from evaluation.config import PROJECT_DIR
from collections import defaultdict

import datasets
from datasets import load_dataset

from training_scripts.utils import normalize_to_likert

data = load_dataset(f"{PROJECT_DIR}/data/granola-eq")

In [42]:
data["train"].to_json("train.jsonl")
data["val"].to_json("val.jsonl")
data["test"].to_json("test.jsonl")

Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

715458

In [2]:
data['train'][0]

{'id': 12642,
 'relation': 'P264',
 'question': 'What music label is On the Radio represented by?',
 'question_entity': 'On the Radio',
 'question_entity_qid': 'Q2482990',
 'question_entity_description': 'single by Natalia',
 'question_entity_popularity': 2371.0,
 'answer': 'Casablanca Records',
 'answer_entity_qid': 'Q1046707',
 'answer_entity_description': 'American record label',
 'answer_entity_popularity': 6638.0,
 'score_for_potential_error': 0.0,
 'granola_answer_1': 'Casablanca Records',
 'granola_answer_2': 'an American record label',
 'granola_answer_3': '',
 'granola_answer_4': '',
 'num_granola_answers': 2}

In [46]:
import pandas as pd

df = pd.DataFrame(data['test'])
entity_levels = defaultdict(set)

for _, row in df.iterrows():
    answers = []
    levels = []
    for i in range(1, 5):
        ans = row[f"granola_answer_{i}"]
        if ans != "":
            answers.append(ans.lower().strip())
            levels.append(i)
    normalized_levels = normalize_to_likert(levels, L=4)
    for answer, level in zip(answers, normalized_levels):
        entity_levels[answer].add(level)

multi_level_entities = {
    entity: levels
    for entity, levels in entity_levels.items()
    if len(levels) > 1
}

len(multi_level_entities)

102

In [21]:
import pandas as pd

rows = []
relation_examples = dict()

for split, items in data.items():
    counter = defaultdict(int)

    for e in items:
        question = e["question"].lower()
        relation_examples[e['relation']] = question
        counter[e['relation']] += 1
        counter["TOTAL"] += 1

    for rel in list(counter.keys()):
        rows.append({
            "relation": rel,
            "split": split,
            "count": counter.get(rel, 0),
        })

In [20]:
df = pd.DataFrame(rows)
df_table = (
    df
    .pivot(index="relation", columns="split", values="count")
    .fillna(0)
    .astype(int)
)
df_table.index = (
    df_table.index
    .str.replace("_", " ")
)
df_table = df_table[["train", "val", "test"]]
df_table = df_table.sort_values(by="train", ascending=False)
latex = df_table.to_latex(
    column_format="lrrr",
    bold_rows=False,
    caption="Distribution of extracted relation types.",
    label="tab:relation-distribution",
)

print(latex)

\begin{table}
\caption{Distribution of extracted relation types.}
\label{tab:relation-distribution}
\begin{tabular}{lrrr}
\toprule
split & train & val & test \\
relation &  &  &  \\
\midrule
TOTAL & 6702 & 1220 & 1220 \\
P20 & 650 & 36 & 49 \\
P19 & 635 & 45 & 64 \\
P69 & 600 & 48 & 30 \\
P276 & 585 & 28 & 55 \\
P159 & 496 & 76 & 67 \\
P26 & 482 & 58 & 142 \\
P131 & 452 & 96 & 61 \\
P176 & 444 & 73 & 124 \\
P50 & 443 & 64 & 104 \\
P170 & 439 & 105 & 56 \\
P264 & 398 & 68 & 95 \\
P127 & 361 & 83 & 147 \\
P40 & 337 & 83 & 127 \\
P112 & 235 & 38 & 57 \\
P175 & 79 & 315 & 35 \\
P740 & 66 & 4 & 7 \\
\bottomrule
\end{tabular}
\end{table}



In [35]:
import numpy as np

word_counter = defaultdict(int)
exists_counter = defaultdict(int)
word = 'England'
max_granola = 4
for e in datasets.concatenate_datasets([data['train'], data['val'], data['test']]):
    answers = []
    granolas = []
    for i in range(1, max_granola + 1):
        answer = e.get(f"granola_answer_{i}")
        if answer:
            answers.append(answer.lower())
            granolas.append(i)
    if not answers:
        continue

    Y_norm = normalize_to_likert(granolas, L=max_granola)
    exists_counter[len(Y_norm)] += 1

    try:
        index = answers.index(word.lower())
        word_counter[Y_norm[index]] += 1
    except ValueError:
        pass

values = []
for key, count in word_counter.items():
    values.extend([key] * count)

values = np.array(values)

mean = values.mean()
variance = values.var()  # population variance by default

print('Word')
print("Mean:", mean)
print("Population variance:", variance)
print(word_counter)

values = []
for key, count in exists_counter.items():
    values.extend([key] * count)

values = np.array(values)

mean = values.mean()
variance = values.var()  # population variance by default

print('\nGran Resolution')
print("Mean:", mean)
print("Population variance:", variance)
print(exists_counter)

Word
Mean: 3.25462
Population variance: 0.44338208
defaultdict(<class 'int'>, {np.float32(4.0): 198, np.float32(2.5): 114, np.float32(3.0): 160, np.float32(1.0): 2, np.float32(2.0): 13})

Gran Resolution
Mean: 2.884270400350033
Population variance: 0.43618005727087455
defaultdict(<class 'int'>, {2: 1966, 3: 5650, 4: 1320, 1: 206})


In [36]:
levels = np.array(list(exists_counter.keys()))
counts = np.array(list(exists_counter.values()))
N = counts.sum()

mean = (levels * counts).sum() / N
variance = ((levels - mean) ** 2 * counts).sum() / N

In [37]:
N = sum(exists_counter.values())

pct_1 = exists_counter.get(1, 0) / N * 100
pct_2 = exists_counter.get(2, 0) / N * 100
pct_3 = exists_counter.get(3, 0) / N * 100
pct_4 = exists_counter.get(4, 0) / N * 100


print(f"~{pct_1:.1f}% have 1 answers overall")
print(f"~{pct_2:.1f}% have 2 answers overall")
print(f"~{pct_3:.1f}% have 3 answers overall")
print(f"~{pct_4:.1f}% have 4 answers overall")

~2.3% have 1 answers overall
~21.5% have 2 answers overall
~61.8% have 3 answers overall
~14.4% have 4 answers overall


In [38]:
from granuscore import GranuScore

granuscore = GranuScore()

[GranuScorer] macOS detected → using FAISS single-thread mode.


In [39]:
from training_scripts.utils import normalize_to_likert
from collections import defaultdict

max_granola = 4
granola_level = defaultdict(list)
for d in data['test']:
    answers = []
    granolas = []
    for i in range(1, max_granola + 1):
        answer = d.get(f"granola_answer_{i}")
        if answer:
            answers.append(answer)
            granolas.append(i)
    if not answers:
        continue

    Y_norm = normalize_to_likert(granolas, L=max_granola)
    for answer, y in zip(answers, Y_norm):
        granola_level[y].append(answer)

In [40]:
for k, v in sorted(granola_level.items()):
    scores = granuscore(v, show_progress_bar=True, encoding_batch_size=128, batch_size=128, split=False)
    print(k, scores.mean(), granuscore.score_to_percentile(scores).mean())

Granuscore: 100%|██████████| 10/10 [00:03<00:00,  2.62it/s]


1.0 1.5523822 28.537469801299366


Granuscore: 100%|██████████| 1/1 [00:00<00:00,  4.35it/s]


2.0 1.9892223 47.32631713067975


Granuscore: 100%|██████████| 6/6 [00:01<00:00,  3.81it/s]


2.5 2.1880107 57.27429697957446


Granuscore: 100%|██████████| 1/1 [00:00<00:00,  4.45it/s]


3.0 2.4084115 64.1238497684135


Granuscore: 100%|██████████| 10/10 [00:01<00:00,  6.12it/s]


4.0 2.791487 77.29311340909707
